# Phase 1 v2 — Dataset Setup and Patient-Level Split Strategy

This notebook replaces the original Phase 1 notebook while leaving it untouched. The key change is that Cleveland is now divided into three internal partitions: **70% train**, **15% validation**, and **15% held-out Cleveland test**. Hungarian and Swiss remain untouched external test sets.

This notebook does **not** generate external output files. The later Python runner will save the processed split files and metadata.

## Updated plan

The processed UCI Cleveland file does not include an explicit patient ID column. Because of that, true subject-level grouping with `GroupShuffleSplit` cannot be performed from the available fields. We therefore document the assumption that each row represents a unique independent patient and use `StratifiedShuffleSplit` to preserve the disease/no-disease ratio across train, validation, and held-out test partitions.

If a future version of the data includes a `patient_id` column, the split should be replaced with `GroupShuffleSplit` so that all rows from the same patient remain in only one partition.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target",
]
RAW_FEATURES = [c for c in COLUMN_NAMES if c != "target"]

SITE_FILES = {
    "cleveland": {
        "filename": "processed.cleveland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
    },
    "hungarian": {
        "filename": "processed.hungarian.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data",
    },
    "swiss": {
        "filename": "processed.switzerland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.switzerland.data",
    },
}

def ensure_raw_files():
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    paths = {}
    for site, info in SITE_FILES.items():
        path = RAW_DATA_DIR / info["filename"]
        paths[site] = path
        if not path.exists() or path.stat().st_size == 0:
            print(f"Downloading {site}...")
            urlretrieve(info["url"], path)
    return paths

def load_site(path, site):
    df = pd.read_csv(path, header=None, names=COLUMN_NAMES, na_values="?")
    for col in COLUMN_NAMES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["target_original"] = df["target"]
    df["target"] = (df["target"] > 0).astype(int)
    df["site"] = site
    df["original_index"] = df.index
    return df

def load_all_sites():
    paths = ensure_raw_files()
    return {site: load_site(path, site) for site, path in paths.items()}

def split_cleveland_v2(cleveland, random_state=RANDOM_STATE):
    # UCI processed Cleveland has no explicit patient ID. Therefore, rows are treated as independent patients.
    # If a patient_id column becomes available later, replace this with GroupShuffleSplit.
    cleveland = cleveland.copy()

    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=random_state)
    rest_idx, test_idx = next(sss_test.split(cleveland, cleveland["target"]))

    rest = cleveland.iloc[rest_idx].copy()
    test = cleveland.iloc[test_idx].copy()

    # Validation is 15% of total. After removing 15% test, validation is 15/85 of the remaining rows.
    val_fraction_of_rest = 0.15 / 0.85
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_fraction_of_rest, random_state=random_state)
    train_rel_idx, val_rel_idx = next(sss_val.split(rest, rest["target"]))

    train = rest.iloc[train_rel_idx].copy()
    validation = rest.iloc[val_rel_idx].copy()

    train["partition"] = "train"
    validation["partition"] = "validation"
    test["partition"] = "test"

    return train, validation, test

def verify_no_overlap(train, validation, test):
    sets = {
        "train": set(train["original_index"]),
        "validation": set(validation["original_index"]),
        "test": set(test["original_index"]),
    }
    assert sets["train"].isdisjoint(sets["validation"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["validation"].isdisjoint(sets["test"])
    print("No Cleveland original_index overlap across train/validation/test.")

def load_phase1_v2_in_memory():
    sites = load_all_sites()
    train, validation, test = split_cleveland_v2(sites["cleveland"])
    verify_no_overlap(train, validation, test)
    return {
        "cleveland_full": sites["cleveland"],
        "cleveland_train": train,
        "cleveland_validation": validation,
        "cleveland_test": test,
        "hungarian": sites["hungarian"],
        "swiss": sites["swiss"],
    }

## Execute — Load data and create the new Cleveland partitions

In [2]:
phase1 = load_phase1_v2_in_memory()

for name, df in phase1.items():
    print(f"{name:22s} shape={df.shape}, positive_rate={df['target'].mean():.2%}")

No Cleveland original_index overlap across train/validation/test.
cleveland_full         shape=(303, 17), positive_rate=45.87%
cleveland_train        shape=(211, 18), positive_rate=45.97%
cleveland_validation   shape=(46, 18), positive_rate=45.65%
cleveland_test         shape=(46, 18), positive_rate=45.65%
hungarian              shape=(294, 17), positive_rate=36.05%
swiss                  shape=(123, 17), positive_rate=93.50%


## Check — Split sizes and stratification

Expected approximate sizes are 70% train, 15% validation, and 15% held-out test. Because 303 rows cannot be split into exact integer percentages, the sizes may differ by one row.

In [3]:
summary_rows = []
for name in ["cleveland_train", "cleveland_validation", "cleveland_test", "hungarian", "swiss"]:
    df = phase1[name]
    summary_rows.append({
        "split": name,
        "rows": len(df),
        "negative": int((df["target"] == 0).sum()),
        "positive": int((df["target"] == 1).sum()),
        "positive_rate": df["target"].mean(),
    })

pd.DataFrame(summary_rows)

,split,rows,negative,positive,positive_rate
0,cleveland_train,211,114,97,0.459716
1,cleveland_validation,46,25,21,0.456522
2,cleveland_test,46,25,21,0.456522
3,hungarian,294,188,106,0.360544
4,swiss,123,8,115,0.934959


## Check — No row overlap inside Cleveland

Because there is no explicit patient ID, overlap is checked using the original Cleveland row index. No original row index should appear in more than one Cleveland partition.

In [4]:
verify_no_overlap(
    phase1["cleveland_train"],
    phase1["cleveland_validation"],
    phase1["cleveland_test"],
)

No Cleveland original_index overlap across train/validation/test.


## Evaluation note

The held-out Cleveland test partition must be treated like a test set and should not be used for imputation, scaling, hyperparameter tuning, calibration, threshold selection, or model selection. It is evaluated only in Phase 4, together with Hungarian and Swiss.